# Personnes traitées pour hypertension artérielle : profil, recours aux soins et coûts

**Master en Sciences Biomédicales, Université de Genève**
Module d'économie de la santé. Travaux pratiques : analyse descriptive de données de remboursement de l'assurance obligatoire des soins.

## Question de recherche

Quel est le profil des personnes traitées pour hypertension artérielle parmi les assuré·es d'un assureur-maladie, comment recourent-elles au système de soins, et quels coûts représentent-elles pour l'assurance obligatoire des soins en comparaison des autres assuré·es ?

## Objectifs

À l'issue de ces travaux pratiques, les étudiant·es seront capables d'interpréter une analyse descriptive simple de données de remboursement, d'en identifier les principales limites, et d'en tirer des éléments d'interprétation à l'intention d'un·e décideur·e de santé publique.

## Données

Les données correspondent à une extraction des décomptes de l'année 2024 d'un assureur-maladie de Suisse romande, portant sur 15 000 assuré·es adultes. Elles sont **entièrement synthétiques** : produites par simulation afin de reproduire les caractéristiques des données réelles, elles ne contiennent aucune information relative à des personnes existantes. La description des variables figure dans le fichier `dictionnaire_donnees.md`.

## Travail attendu

Chaque groupe rédige une note d'une page destinée à un·e responsable de santé publique, présentant la population concernée, les traitements et le recours aux soins, les coûts associés, deux ou trois messages principaux et une limite d'interprétation.

## Organisation du document

Les cellules s'exécutent dans l'ordre (Maj + Entrée). Le code est fourni ; sa compréhension détaillée n'est pas requise, les commentaires (lignes précédées de `#`) en indiquent l'objet. Chaque section se termine par des questions destinées à la discussion en groupe, dont les réponses alimentent la note. Les paramètres pouvant être modifiés sont signalés comme tels ; les analyses complémentaires sont facultatives.

| Section | Contenu | Durée indicative |
|---|---|---|
| 1 | Description des données | 15 min |
| 2 | Contrôle de qualité et préparation des données | 15 min |
| 3 | Identification des personnes traitées | 15 min |
| 4 | Caractéristiques des personnes traitées | 15 min |
| 5 | Traitements antihypertenseurs | 15 min |
| 6 | Recours aux soins | 15 min |
| 7 | Coûts | 20 min |
| 8 | Synthèse et rédaction de la note | 40 min |

In [ ]:
# @title Préparation de l'environnement (à exécuter sans modification)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", "'") if abs(x) >= 100 else f"{x:.1f}")

ANNEE = 2024
GROUPES = ["HTA traitée", "Autres assuré·es"]
COULEURS = {"HTA traitée": "#C0392B", "Autres assuré·es": "#95A5A6"}


def axe_pct(ax, axe="y"):
    """Affiche un axe en pourcentage."""
    (ax.yaxis if axe == "y" else ax.xaxis).set_major_formatter(mtick.PercentFormatter(1))


def axe_chf(ax, axe="y"):
    """Affiche un axe en CHF avec séparateur de milliers."""
    formateur = mtick.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", "'"))
    (ax.yaxis if axe == "y" else ax.xaxis).set_major_formatter(formateur)


print("Environnement prêt.")

In [ ]:
# @title Chargement des données
# Enseignant·e : adresse du dossier contenant les fichiers CSV (ex. dépôt GitHub, lien « raw »)
URL_DONNEES = "https://raw.githubusercontent.com/daderidd/heco-tp-hypertension/main/donnees/"


def lire(nom, **options):
    local = Path("donnees") / nom                       # dossier « donnees » déposé dans Colab
    source = local if local.exists() else URL_DONNEES + nom
    return pd.read_csv(source, **options)


assures = lire("assures.csv", parse_dates=["debut_couverture", "fin_couverture"])
prestations = lire("prestations.csv", parse_dates=["date_prestation", "date_decompte"])
medicaments = lire("medicaments.csv", parse_dates=["date_delivrance"])
referentiel = lire("referentiel_atc.csv")

print(f"assures      : {len(assures):>8,} lignes")
print(f"prestations  : {len(prestations):>8,} lignes")
print(f"medicaments  : {len(medicaments):>8,} lignes")

---
## 1. Description des données

Les données comprennent trois tables, reliées par l'identifiant `id_assure` :

```
assures       1 ligne par personne assurée
   ├── prestations   1 ligne par position de facture (médecin, labo, hôpital...)
   └── medicaments   1 ligne par délivrance en pharmacie
```

In [ ]:
# Les 5 premières lignes de chaque table
display(assures.head())
display(prestations.head())
display(medicaments.head())

In [ ]:
# Âge au 31 décembre, puis distribution par sexe
assures["age"] = ANNEE - assures["annee_naissance"]

ax = sns.histplot(data=assures, x="age", hue="sexe", binwidth=5, multiple="stack")
ax.set(title="Âge des assuré·es", xlabel="Âge (ans)", ylabel="Nombre de personnes");

In [ ]:
# Nombre de lignes par type de prestation
ax = prestations["type_prestation"].value_counts().plot.barh(color="#34495E")
ax.invert_yaxis()
ax.set(title="Lignes de prestations par type", xlabel="Nombre de lignes", ylabel="");

**Questions**
1. Quelle est l'unité d'observation de chacune des trois tables ?
2. Quelle information, nécessaire à l'étude de l'hypertension, ces données ne contiennent-elles pas ?
3. La distribution des âges présente-t-elle des valeurs invraisemblables ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 2. Contrôle de qualité et préparation des données

Les données administratives comportent fréquemment des erreurs et des incohérences liées aux processus de facturation. Les contrôles suivants précèdent toute analyse.

In [ ]:
colonnes_prest = [c for c in prestations.columns if c != "id_ligne"]   # tout sauf l'identifiant de ligne
colonnes_med = [c for c in medicaments.columns if c != "id_ligne"]

controles = pd.Series({
    "Âge > 105 ans": (assures["age"] > 105).sum(),
    "Sexe manquant": assures["sexe"].isna().sum(),
    "Canton manquant": assures["canton"].isna().sum(),
    "Prestations : lignes en double": prestations.duplicated(subset=colonnes_prest).sum(),
    "Prestations : montant négatif": (prestations["cout_chf"] < 0).sum(),
    "Prestations : montant nul": (prestations["cout_chf"] == 0).sum(),
    f"Prestations : date hors {ANNEE}": (prestations["date_prestation"].dt.year != ANNEE).sum(),
    "Médicaments : lignes en double": medicaments.duplicated(subset=colonnes_med).sum(),
    "Médicaments : code ATC mal écrit": (medicaments["code_atc"] != medicaments["code_atc"].str.strip().str.upper()).sum(),
}, name="Nombre")
controles.to_frame()

In [ ]:
# Exemple : lignes d'une même personne, même date et même type de prestation qu'une ligne à montant négatif
cas = prestations[prestations["cout_chf"] < 0].iloc[0]
prestations[(prestations["id_assure"] == cas["id_assure"])
            & (prestations["date_prestation"] == cas["date_prestation"])
            & (prestations["type_prestation"] == cas["type_prestation"])]

**Question.** Comment interpréter la succession de lignes ci-dessus ? Quelles conséquences aurait un dénombrement des lignes pour mesurer le nombre de consultations ?

**Règles de préparation retenues.** Les tables d'origine ne sont pas modifiées ; les analyses portent sur des copies. Les âges supérieurs à 105 ans sont considérés comme manquants, les doublons exacts sont supprimés, les prestations antérieures à 2024 sont exclues et les codes ATC sont normalisés. Les lignes d'annulation sont conservées dans le calcul des montants, puisqu'elles compensent la facture d'origine. Pour dénombrer les recours, les lignes d'une même personne, à une même date et pour un même type de prestation, sont regroupées en un **contact** lorsque leur montant net est positif.

In [ ]:
# Assuré·es
assures_ok = assures.copy()
assures_ok.loc[assures_ok["age"] > 105, "age"] = np.nan
assures_ok["classe_age"] = pd.cut(assures_ok["age"], bins=[18, 39, 49, 59, 69, 79, 110],
                                  labels=["19-39", "40-49", "50-59", "60-69", "70-79", "80+"])

# Médicaments
med_ok = medicaments.drop_duplicates(subset=colonnes_med).copy()
med_ok["code_atc"] = med_ok["code_atc"].str.strip().str.upper()

# Prestations
prest_ok = prestations.drop_duplicates(subset=colonnes_prest)
prest_ok = prest_ok[prest_ok["date_prestation"].dt.year == ANNEE].copy()

# Contacts : une personne, un jour, un type de prestation, montant net positif
contacts = (prest_ok
            .groupby(["id_assure", "date_prestation", "type_prestation"], as_index=False)
            .agg(cout_chf=("cout_chf", "sum"), specialite=("specialite", "first")))
contacts = contacts[contacts["cout_chf"] > 0]

print(f"Prestations : {len(prestations):,} lignes → {len(prest_ok):,} lignes nettoyées → {len(contacts):,} contacts")
print(f"Médicaments : {len(medicaments):,} lignes → {len(med_ok):,} délivrances")

Le nombre de lignes de facturation ne correspond donc pas au nombre de recours aux soins.

---
## 3. Identification des personnes traitées pour hypertension

Les données de facturation ambulatoire ne comportent pas de diagnostic. Les personnes traitées sont donc identifiées à partir des médicaments délivrés, selon la définition opérationnelle suivante : **au moins deux délivrances en 2024 d'un antihypertenseur**, c'est-à-dire d'un médicament dont le code ATC commence par l'un des préfixes ci-dessous. Les personnes ainsi identifiées forment le groupe « HTA traitée » (hypertension artérielle traitée).

| C02 | C03 | C07 | C08 | C09 |
|---|---|---|---|---|
| action centrale / alpha-bloquants | diurétiques | bêtabloquants | inhibiteurs calciques | IEC / sartans (ARA) |

In [ ]:
# Paramètres de la définition opérationnelle (modifiables, voir les analyses complémentaires)
CLASSES_ANTIHYPERTENSEURS = ["C02", "C03", "C07", "C08", "C09"]
SEUIL_DELIVRANCES = 2

In [ ]:
antihta = med_ok[med_ok["code_atc"].str[:3].isin(CLASSES_ANTIHYPERTENSEURS)]
nb_delivrances = antihta.groupby("id_assure").size()
ids_hta = nb_delivrances[nb_delivrances >= SEUIL_DELIVRANCES].index

assures_ok["groupe"] = np.where(assures_ok["id_assure"].isin(ids_hta), "HTA traitée", "Autres assuré·es")
hta = assures_ok["groupe"] == "HTA traitée"

print(f"{hta.sum():,} personnes traitées pour hypertension, soit {hta.mean():.1%} des assuré·es")

In [ ]:
ax = nb_delivrances.clip(upper=20).value_counts().sort_index().plot.bar(color="#34495E", width=0.8)
ax.axvline(SEUIL_DELIVRANCES - 1.5, color="#C0392B", ls="--")
ax.set(title="Nombre de délivrances d'antihypertenseurs par personne (20 = 20 ou plus)",
       xlabel="Délivrances en 2024", ylabel="Nombre de personnes")
ax.tick_params(axis="x", rotation=0);

Les bêtabloquants et les diurétiques ont d'autres indications que l'hypertension. Le tableau suivant décrit les personnes identifiées sur la base d'une seule classe thérapeutique, bêtabloquant (C07) ou diurétique de l'anse (C03C).

In [ ]:
classes_par_personne = antihta.groupby("id_assure")["code_atc"].agg(lambda s: set(s.str[:3]))
une_seule_classe = classes_par_personne[classes_par_personne.apply(len) == 1].index.intersection(ids_hta)

a_verifier = (antihta[antihta["id_assure"].isin(une_seule_classe)
                      & antihta["code_atc"].str.startswith(("C07", "C03C"))]
              .drop_duplicates(["id_assure", "code_atc"])
              .merge(referentiel[["code_atc", "substance"]], on="code_atc")
              .merge(assures_ok[["id_assure", "age", "sexe"]], on="id_assure"))

anticoagules = med_ok.loc[med_ok["code_atc"].str.startswith("B01AF"), "id_assure"].unique()
a_verifier["aussi_anticoagulant"] = a_verifier["id_assure"].isin(anticoagules)

(a_verifier.groupby("substance")
 .agg(personnes=("id_assure", "nunique"), age_median=("age", "median"),
      pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
      pct_avec_anticoagulant=("aussi_anticoagulant", lambda s: 100 * s.mean()))
 .sort_values("personnes", ascending=False))

**Questions**
1. La proportion de personnes identifiées est-elle plausible ?
2. Quelles personnes cette définition omet-elle, et lesquelles inclut-elle à tort ? On pourra considérer l'âge et le sexe des personnes traitées par propranolol, ainsi que l'association d'un diurétique et d'un anticoagulant.
3. Cette proportion constitue-t-elle une estimation de la prévalence de l'hypertension ?

**Analyse complémentaire.** Modifier `SEUIL_DELIVRANCES` (par exemple 1 ou 4), puis exécuter les cellules suivantes (menu *Exécution*). Quelle est la sensibilité des résultats à la définition retenue ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 4. Caractéristiques des personnes traitées

Les autres maladies chroniques sont, elles aussi, approchées par les médicaments délivrés : antidiabétiques (A10), statines (C10) et anticoagulants oraux directs (B01AF).

In [ ]:
prevalence = assures_ok.groupby(["classe_age", "sexe"], observed=True)["groupe"].apply(lambda g: (g == "HTA traitée").mean())
ax = prevalence.reset_index(name="part").pipe(
    lambda d: sns.barplot(data=d, x="classe_age", y="part", hue="sexe", palette={"F": "#8E44AD", "M": "#2980B9"}))
axe_pct(ax)
ax.set(title="Part des personnes traitées pour hypertension", xlabel="Classe d'âge", ylabel="");

In [ ]:
# Indicateurs de comorbidité fondés sur les délivrances
PROXYS = {"Diabète (A10)": "A10", "Statine (C10)": "C10", "Anticoagulant (B01AF)": "B01AF"}
for nom, prefixe in PROXYS.items():
    assures_ok[nom] = assures_ok["id_assure"].isin(med_ok.loc[med_ok["code_atc"].str.startswith(prefixe), "id_assure"])

profil = assures_ok.groupby("groupe").agg(
    personnes=("id_assure", "size"),
    age_moyen=("age", "mean"),
    pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
    pct_subside=("subside", lambda s: 100 * s.mean()),
    **{f"% {nom}": (nom, lambda s: 100 * s.mean()) for nom in PROXYS},
).T[GROUPES]
profil

In [ ]:
comorbidites = assures_ok.groupby("groupe")[list(PROXYS)].mean().T[GROUPES]
ax = comorbidites.plot.barh(color=COULEURS, width=0.75)
ax.invert_yaxis()
axe_pct(ax, "x")
ax.set(title="Autres traitements chroniques", xlabel="Part des personnes", ylabel="");

La cellule suivante compare la distribution d'une variable entre les deux groupes. Paramètre modifiable : `VARIABLE` (`franchise_chf`, `modele_assurance`, `canton`, `subside` ou `classe_age`).

In [ ]:
VARIABLE = "franchise_chf"   # paramètre modifiable

repartition = pd.crosstab(assures_ok["groupe"], assures_ok[VARIABLE], normalize="index").loc[GROUPES]
ax = repartition.plot.barh(stacked=True, colormap="viridis", figsize=(8, 2.8), width=0.7)
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(title=VARIABLE, bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title=f"Répartition selon « {VARIABLE} »", xlabel="", ylabel="");

**Questions**
1. Comment caractériser, en deux phrases, les personnes traitées pour hypertension ?
2. Comment expliquer la fréquence plus élevée de la franchise minimale dans ce groupe ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 5. Traitements antihypertenseurs

Certains médicaments sont des associations fixes, qui réunissent plusieurs classes thérapeutiques sous un seul code ATC (par exemple `C09DA` : antagoniste de l'angiotensine II et diurétique). Chaque code est donc converti en la ou les classes qu'il contient.

In [ ]:
CLASSES = {   # début du code ATC → classes contenues
    "C09AA": ["IEC"], "C09BA": ["IEC", "Diurétique"], "C09BB": ["IEC", "Inhibiteur calcique"],
    "C09CA": ["Sartan (ARA)"], "C09DA": ["Sartan (ARA)", "Diurétique"],
    "C09DB": ["Sartan (ARA)", "Inhibiteur calcique"],
    "C09DX": ["Sartan (ARA)", "Inhibiteur calcique", "Diurétique"],
    "C08": ["Inhibiteur calcique"], "C07": ["Bêtabloquant"], "C02": ["Autre antihypertenseur"],
    "C03A": ["Diurétique"], "C03B": ["Diurétique"], "C03C": ["Diurétique de l'anse"], "C03D": ["Antagoniste aldostérone"],
}


def classes_du_code(code_atc):
    for debut in sorted(CLASSES, key=len, reverse=True):
        if code_atc.startswith(debut):
            return CLASSES[debut]
    return ["Non classé"]


trt = antihta[antihta["id_assure"].isin(ids_hta)].copy()
trt["classe"] = trt["code_atc"].apply(classes_du_code)
trt["combinaison_fixe"] = trt["classe"].apply(len) > 1
trt_classes = trt.explode("classe")

par_personne = pd.DataFrame({
    "nb_codes": trt.groupby("id_assure")["code_atc"].nunique(),
    "nb_classes": trt_classes.groupby("id_assure")["classe"].nunique(),
    "combinaison_fixe": trt.groupby("id_assure")["combinaison_fixe"].any(),
})
print(f"{par_personne['combinaison_fixe'].mean():.0%} des personnes reçoivent au moins une combinaison fixe")

In [ ]:
part_classes = trt_classes.groupby("classe")["id_assure"].nunique().div(len(par_personne)).sort_values()
ax = part_classes.plot.barh(color=COULEURS["HTA traitée"])
axe_pct(ax, "x")
ax.set(title="Classes utilisées (au moins une fois en 2024)", xlabel="Part des personnes traitées", ylabel="");

In [ ]:
comparaison = pd.DataFrame({
    "Compté en codes ATC": par_personne["nb_codes"].clip(upper=4).value_counts(normalize=True),
    "Compté en classes": par_personne["nb_classes"].clip(upper=4).value_counts(normalize=True),
}).sort_index().rename(index={4: "4+"})
ax = comparaison.plot.bar(color=["#BDC3C7", COULEURS["HTA traitée"]], width=0.75)
axe_pct(ax)
ax.tick_params(axis="x", rotation=0)
ax.set(title="Nombre de médicaments antihypertenseurs par personne", xlabel="Nombre", ylabel="Part des personnes");

In [ ]:
par_personne = par_personne.join(assures_ok.set_index("id_assure")["classe_age"])
polytherapie = pd.crosstab(par_personne["classe_age"], par_personne["nb_classes"].clip(upper=3), normalize="index")
ax = polytherapie.rename(columns={1: "1 classe", 2: "2 classes", 3: "3 classes ou +"}).plot.bar(
    stacked=True, color=["#F5B7B1", "#E74C3C", "#922B21"], width=0.75)
axe_pct(ax)
ax.tick_params(axis="x", rotation=0)
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Nombre de classes selon l'âge", xlabel="Classe d'âge", ylabel="");

**Questions**
1. Quelles sont les classes les plus utilisées ? La monothérapie est-elle majoritaire, et comment évolue-t-elle avec l'âge ?
2. Pourquoi un dénombrement des codes ATC sous-estime-t-il la fréquence des associations thérapeutiques ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 6. Recours aux soins

Le recours aux soins est mesuré par le nombre de contacts (définis à la section 2) par personne et par type de prestation.

In [ ]:
recours = (pd.crosstab(contacts["id_assure"], contacts["type_prestation"])
           .reindex(assures_ok["id_assure"], fill_value=0))
recours["hospitalise"] = recours["stationnaire"] > 0
recours = recours.join(assures_ok.set_index("id_assure")[["groupe", "classe_age"]])

moyennes = recours.groupby("groupe")[["medecin_generaliste", "medecin_specialiste", "laboratoire",
                                      "ambulatoire_hospitalier", "physiotherapie"]].mean().T[GROUPES]
ax = moyennes.plot.barh(color=COULEURS, width=0.75)
ax.invert_yaxis()
ax.set(title="Nombre moyen de contacts par personne en 2024", xlabel="Contacts", ylabel="");

In [ ]:
print("Part des personnes hospitalisées au moins une fois :")
recours.groupby("groupe")["hospitalise"].mean().loc[GROUPES].map("{:.1%}".format).to_frame("")

Les personnes traitées étant en moyenne plus âgées, la comparaison est répétée par classe d'âge afin de distinguer l'effet de l'âge de celui de la condition étudiée.

In [ ]:
ax = sns.pointplot(data=recours, x="classe_age", y="medecin_generaliste", hue="groupe",
                   palette=COULEURS, hue_order=GROUPES, errorbar=None)
ax.set(title="Consultations chez le généraliste, à âge égal", xlabel="Classe d'âge", ylabel="Contacts moyens par personne")
ax.legend(title="");

In [ ]:
specialites = (contacts[contacts["type_prestation"] == "medecin_specialiste"]
               .merge(assures_ok[["id_assure", "groupe"]], on="id_assure"))
repartition = pd.crosstab(specialites["specialite"], specialites["groupe"], normalize="columns")[GROUPES]
ax = repartition.sort_values("HTA traitée").plot.barh(color=COULEURS, width=0.75, figsize=(8, 5))
axe_pct(ax, "x")
ax.set(title="Répartition des consultations de spécialistes", xlabel="Part des consultations", ylabel="");

**Questions**
1. Les personnes traitées recourent-elles davantage aux soins, et pour quels types de prestations ?
2. La différence persiste-t-elle à âge égal ? Quels autres facteurs pourraient l'expliquer ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 7. Coûts

Le coût est défini comme le montant décompté à la charge de l'assurance obligatoire des soins en 2024, prestations et médicaments confondus. Pour les séjours hospitaliers, seule la part financée par l'assureur figure dans les données.

In [ ]:
couts = prest_ok.pivot_table(index="id_assure", columns="type_prestation", values="cout_chf", aggfunc="sum")
couts["med_antihypertenseurs"] = antihta.groupby("id_assure")["cout_chf"].sum()
couts["med_autres"] = med_ok[~med_ok.index.isin(antihta.index)].groupby("id_assure")["cout_chf"].sum()
couts = couts.reindex(assures_ok["id_assure"]).fillna(0)

POSTES = list(couts.columns)
couts["total"] = couts[POSTES].sum(axis=1)
couts = couts.join(assures_ok.set_index("id_assure")[["groupe", "classe_age", "franchise_chf", "motif_fin_couverture"]])

(couts.groupby("groupe")["total"]
 .describe(percentiles=[0.5, 0.9])[["count", "mean", "50%", "90%", "max"]]
 .rename(columns={"count": "personnes", "mean": "moyenne", "50%": "médiane", "90%": "90e centile"})
 .loc[GROUPES].round(0))

In [ ]:
ax = sns.histplot(data=couts[couts["total"] > 0], x="total", hue="groupe", hue_order=GROUPES,
                  palette=COULEURS, log_scale=True, bins=40, stat="density", common_norm=False, element="step")
ax.set(title="Distribution du coût annuel par personne (échelle logarithmique)", xlabel="CHF", ylabel="");

In [ ]:
# Concentration des coûts : part cumulée des coûts selon la part cumulée des assuré·es
tri = couts.sort_values("total", ascending=False)
part_personnes = np.arange(1, len(tri) + 1) / len(tri)
part_couts = tri["total"].cumsum() / tri["total"].sum()
top10 = part_couts.iloc[int(0.1 * len(tri)) - 1]

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(part_personnes, part_couts, color="#34495E", lw=2)
ax.plot([0, 1], [0, 1], color="grey", ls=":")
ax.axvline(0.1, color="#C0392B", ls="--")
ax.annotate(f"10 % des assuré·es\n= {top10:.0%} des coûts", xy=(0.1, top10), xytext=(0.25, 0.45),
            arrowprops=dict(arrowstyle="->"))
axe_pct(ax); axe_pct(ax, "x")
ax.set(title="Concentration des coûts", xlabel="Assuré·es, du plus au moins coûteux", ylabel="Part cumulée des coûts");

print(f"Part des personnes traitées pour HTA parmi les 10 % les plus coûteux : {(tri['groupe'].iloc[:len(tri)//10] == 'HTA traitée').mean():.0%}")

In [ ]:
NOMS = {"stationnaire": "Hospitalisation", "ambulatoire_hospitalier": "Ambulatoire hospitalier",
        "medecin_generaliste": "Généraliste", "medecin_specialiste": "Spécialiste", "laboratoire": "Laboratoire",
        "physiotherapie": "Physiothérapie", "soins_domicile": "Soins à domicile",
        "med_antihypertenseurs": "Médicaments : antihypertenseurs", "med_autres": "Médicaments : autres"}
postes = couts.groupby("groupe")[POSTES].mean().loc[GROUPES]
postes = postes[postes.sum().sort_values(ascending=False).index]
ax = postes.rename(columns=NOMS).plot.bar(stacked=True, colormap="tab10", width=0.6, figsize=(7, 5))
axe_chf(ax)
ax.tick_params(axis="x", rotation=0)
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Coût moyen par personne et par poste", xlabel="", ylabel="CHF");

print(f"Antihypertenseurs = {postes.loc['HTA traitée', 'med_antihypertenseurs'] / postes.loc['HTA traitée'].sum():.1%} "
      "du coût moyen des personnes traitées")

In [ ]:
ax = sns.pointplot(data=couts, x="classe_age", y="total", hue="groupe", palette=COULEURS,
                   hue_order=GROUPES, estimator="mean", errorbar=None)
axe_chf(ax)
ax.set(title="Coût moyen par personne, à âge égal", xlabel="Classe d'âge", ylabel="CHF")
ax.legend(title="");

In [ ]:
poids = couts.groupby("groupe").agg(personnes=("total", "size"), couts=("total", "sum"))
poids = (poids / poids.sum()).loc[GROUPES]
ax = poids.T.rename(index={"personnes": "Part des assuré·es", "couts": "Part des coûts"}).plot.barh(
    stacked=True, color=COULEURS, width=0.6, figsize=(8, 2.5))
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Part des personnes traitées dans l'effectif et dans les coûts", xlabel="", ylabel="");

**Questions**
1. Compte tenu de la distribution des coûts, quel indicateur (moyenne ou médiane) convient-il de présenter à un·e décideur·e, et dans quel but ?
2. Quelle part des coûts des personnes traitées est attribuable aux antihypertenseurs ?
3. L'écart de coût persiste-t-il à âge égal ? Peut-on l'attribuer à l'hypertension ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### Analyses complémentaires

In [ ]:
# Coût selon la franchise
ax = sns.barplot(data=couts, x="franchise_chf", y="total", hue="groupe", palette=COULEURS,
                 hue_order=GROUPES, estimator="mean", errorbar=None)
axe_chf(ax)
ax.set(title="Coût moyen selon la franchise", xlabel="Franchise (CHF)", ylabel="CHF")
ax.legend(title="");

**Question.** Les coûts observés des personnes ayant choisi une franchise de 2 500 CHF sont très faibles. Quels mécanismes peuvent l'expliquer, et lequel relève d'un artefact de mesure ?

In [ ]:
# Personnes non assurées toute l'année
couts["motif"] = couts["motif_fin_couverture"].fillna("assuré·e toute l'année")
couts.groupby("motif")["total"].agg(["size", "mean", "median"]).round(0)

**Question.** Que suggèrent ces résultats concernant les coûts en fin de vie ? Quel biais introduit la comparaison de personnes assurées pendant des durées différentes ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 8. Synthèse et rédaction de la note

Le tableau suivant rassemble les principaux résultats. Les valeurs dépendent des choix effectués dans les sections précédentes.

In [ ]:
# Synthèse des principaux résultats
g = couts.groupby("groupe")["total"]
chiffres = pd.Series({
    "Personnes traitées pour hypertension": f"{hta.sum():,} ({hta.mean():.1%} des assuré·es)",
    "Âge moyen (traitées / autres)": f"{profil.loc['age_moyen', 'HTA traitée']:.0f} / {profil.loc['age_moyen', 'Autres assuré·es']:.0f} ans",
    "Part de femmes parmi les traitées": f"{profil.loc['pct_femmes', 'HTA traitée']:.0f}%",
    "Traitées avec ≥ 2 classes": f"{(par_personne['nb_classes'] >= 2).mean():.0%}",
    "Généraliste, contacts/an (traitées / autres)":
        f"{moyennes.loc['medecin_generaliste', 'HTA traitée']:.1f} / {moyennes.loc['medecin_generaliste', 'Autres assuré·es']:.1f} par an",
    "Hospitalisées ≥ 1 fois (traitées / autres)":
        f"{recours.groupby('groupe')['hospitalise'].mean()['HTA traitée']:.0%} / {recours.groupby('groupe')['hospitalise'].mean()['Autres assuré·es']:.0%}",
    "Coût moyen (traitées / autres)": f"{g.mean()['HTA traitée']:,.0f} / {g.mean()['Autres assuré·es']:,.0f} CHF",
    "Coût médian (traitées / autres)": f"{g.median()['HTA traitée']:,.0f} / {g.median()['Autres assuré·es']:,.0f} CHF",
    "Part des coûts totaux portée par les traitées": f"{poids.loc['HTA traitée', 'couts']:.0%}",
    "Part des antihypertenseurs dans leur coût": f"{postes.loc['HTA traitée', 'med_antihypertenseurs'] / postes.loc['HTA traitée'].sum():.1%}",
}).str.replace(",", "'")
chiffres.to_frame("Valeur")

### Structure de la note

**Destinataire :** responsable de la santé publique cantonale.
**Objet :** personnes traitées pour hypertension artérielle : profil, recours aux soins et coûts.

1. **Population concernée.** Effectif, proportion et principales caractéristiques.
2. **Traitements et recours aux soins.** Éléments distinguant les personnes traitées des autres assuré·es.
3. **Coûts associés.** Ordres de grandeur et principaux déterminants.
4. **Messages principaux.** Deux ou trois, formulés en vue de la décision.
5. **Limite d'interprétation.** La principale limite et ses implications pour la décision.

La note tient sur une page et comporte au plus un graphique. Les chiffres sont arrondis et le vocabulaire technique propre à l'analyse des données est évité.